# 002 · Tuples

> **Syntax › Collections** · Python 3.14

A `tuple` is an ordered, **immutable** sequence. Think of it as a lightweight, read-only record of a fixed number of values — the closest C# relatives are `ValueTuple` (`(int, string)`) and `System.Tuple<T1, T2>`.

**In this notebook you will learn:**

- how to create tuples, including the single-element `(x,)` gotcha
- why parentheses are often optional — the comma makes the tuple
- how tuples are used to return multiple values and as dictionary keys
- why tuple immutability is only *shallow*
- `namedtuple` and `typing.NamedTuple` — tuples with named fields

## C# mental model

| C# | Python |
|---|---|
| `(int, string)` (`ValueTuple`) | `tuple[int, str]` (type hint only) |
| `(1, "a")` | `(1, "a")` or just `1, "a"` |
| `Tuple.Create(1, "a")` / `tuple.Item1` | `t[0]` |
| `(int X, int Y)` named elements | `namedtuple` / `typing.NamedTuple` |
| `record struct Point(int X, int Y)` | `typing.NamedTuple` (or a dataclass) |

Two key differences:

- C#'s `ValueTuple` is **mutable** (its fields are public fields — `t.Item1 = 5` compiles). A Python tuple can never be changed after creation.
- A Python tuple can be of **any length** — it is a sequence, so you can index, slice, iterate and `len()` it.

## Creating tuples

```csharp
// C#
var point = (3, 4);
(int, string) pair = (1, "one");
var empty = ValueTuple.Create();   // rarely used
```

In [ ]:
point = (3, 4)
pair = (1, "one")
empty = ()
from_list = tuple([1, 2, 3])     # convert any iterable
from_string = tuple("abc")

print(point, pair, empty, from_list, from_string)
print(type(point).__name__, len(point))

## The comma makes the tuple — parentheses are optional

In Python it is the **comma**, not the parentheses, that creates a tuple. Parentheses are only needed for grouping (or when the tuple would otherwise be ambiguous, e.g. as a function argument).

In [ ]:
coords = 10, 20          # a tuple, no parentheses needed
print(coords, type(coords).__name__)

x, y = coords            # unpacking (see Collections › 006 · Unpacking)
print(x, y)

⚠️ **Gotcha:** a single-element tuple needs a **trailing comma**. `(42)` is just the integer 42 in parentheses.

In [ ]:
not_a_tuple = (42)
single = (42,)
also_single = 42,

print(type(not_a_tuple).__name__)   # int
print(type(single).__name__)        # tuple
print(also_single)                  # (42,)

⚠️ **Gotcha:** the flip side — a stray trailing comma silently turns a value into a tuple. This is a common bug when copying code from a dict literal or argument list:

In [ ]:
total = 99.5,          # oops — trailing comma
print(total)           # (99.5,) — a tuple, not a float

## Reading tuples: indexing, slicing, iterating

Tuples support everything a read-only sequence does: indexing (including negative), slicing, `len`, `in`, iteration, and the two methods `count` and `index`.

In [ ]:
rgb = (255, 128, 0, 128)

print(rgb[0], rgb[-1])        # indexing — C#: rgb.Item1 (no index-from-end on ValueTuple)
print(rgb[1:3])               # slicing returns a new tuple
print(128 in rgb)             # membership
print(rgb.count(128))         # 2 — how many times the value occurs
print(rgb.index(128))         # 1 — first position (ValueError if missing)

for channel in rgb:
    print(channel, end=" ")

## Immutability

You cannot assign to an element, append, or remove. There are no mutating methods at all.

In [ ]:
point = (3, 4)
try:
    point[0] = 99
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(hasattr(point, "append"))   # False

To "change" a tuple you build a new one — the same pattern as C# `with` expressions on records:

```csharp
// C#
var moved = point with { X = 99 };   // for a record
```

In [ ]:
point = (3, 4)
moved = (99,) + point[1:]       # concatenation creates a new tuple
print(point, moved)

⚠️ **Gotcha: immutability is shallow.** A tuple holds *references*. The tuple itself can't be re-pointed, but if one of its elements is a mutable object (like a list), that object can still change — just as a C# `readonly` field holding a `List<T>` only protects the reference.

In [ ]:
team = ("alice", ["python", "sql"])
team[1].append("rust")         # mutating the list *inside* the tuple — allowed
print(team)

try:
    team[1] = ["go"]           # replacing the element — not allowed
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Returning multiple values

Tuples are how Python functions return several values. The syntax is the same as C# tuple returns with deconstruction.

```csharp
// C#
static (int Min, int Max) MinMax(IEnumerable<int> xs) => (xs.Min(), xs.Max());

var (lo, hi) = MinMax([3, 1, 4]);
```

In [ ]:
def min_max(values: list[int]) -> tuple[int, int]:
    return min(values), max(values)    # returns a tuple

result = min_max([3, 1, 4, 1, 5])
print(result)

lo, hi = min_max([3, 1, 4, 1, 5])     # unpack straight away
print(lo, hi)

## Tuples as dictionary keys (hashability)

Because tuples are immutable, they are **hashable** (as long as all their elements are hashable), so they can be used as `dict` keys and `set` members. Lists cannot.

In C# you'd use a `ValueTuple` as a composite key: `Dictionary<(int, int), string>`.

In [ ]:
grid = {
    (0, 0): "start",
    (2, 3): "treasure",
}
print(grid[(2, 3)])
print(grid[2, 3])          # parentheses optional inside [] too

try:
    {[0, 0]: "start"}      # list as key
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

A tuple that *contains* a list is not hashable either — hashability requires every element to be hashable:

In [ ]:
print(hash((1, 2, "a")))   # works (the exact number varies)

try:
    hash((1, [2, 3]))
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Equality and comparison

Tuples compare **by value**, element by element, and order **lexicographically** — like C# `ValueTuple` equality, plus ordering for free. This makes tuples great sort keys.

In [ ]:
print((1, 2) == (1, 2))          # True
print((1, 2) < (1, 3))           # True — compares first elements, then second...
print((2, 0) > (1, 99))          # True

people = [("bob", 25), ("alice", 30), ("carol", 25)]
print(sorted(people, key=lambda p: (p[1], p[0])))   # by age, then name

## Named tuples: `collections.namedtuple`

Index-based access (`t[0]`) gets unreadable fast. `namedtuple` creates a tuple *subclass* whose elements also have names — similar to C#'s named tuple elements `(int X, int Y)`, but the names are part of a real type.

```csharp
// C#
(int X, int Y) p = (3, 4);
Console.WriteLine(p.X);
```

In [ ]:
from collections import namedtuple

Point = namedtuple("Point", ["x", "y"])

p = Point(3, 4)
print(p)                 # nice repr: Point(x=3, y=4)
print(p.x, p[1])         # access by name or index
print(p == (3, 4))       # still a tuple — compares equal to a plain tuple

Named tuples are still immutable. Use `_replace` to get a modified copy (like a C# `with` expression), and `_asdict` to convert to a dict:

In [ ]:
p = Point(3, 4)
q = p._replace(x=10)
print(p, q)
print(q._asdict())

## Typed named tuples: `typing.NamedTuple`

`typing.NamedTuple` is the modern, class-based syntax. It supports type hints, default values, docstrings and methods — making it the closest analogue to a C# positional `record struct`.

```csharp
// C#
public readonly record struct Product(string Name, decimal Price, int Qty = 1)
{
    public decimal Total => Price * Qty;
}
```

In [ ]:
from typing import NamedTuple

class Product(NamedTuple):
    name: str
    price: float
    qty: int = 1                  # default value

    @property
    def total(self) -> float:
        return self.price * self.qty

item = Product("widget", 2.5, qty=4)
print(item)
print(item.total)

name, price, qty = item          # it's still a tuple, so it unpacks
print(name, price, qty)

⚠️ **Gotcha:** because a `NamedTuple` *is* a tuple, it compares equal to any tuple with the same values — even a different `NamedTuple` type. A C# `record` would compare type as well. If you need type-aware equality (or mutability), use a dataclass instead *(see Classes & OOP › 008 · Dataclasses)*.

In [ ]:
class Point2D(NamedTuple):
    x: int
    y: int

class Size(NamedTuple):
    width: int
    height: int

print(Point2D(1, 2) == Size(1, 2))   # True — probably not what you want

## When to use a tuple vs a list

| Use a **tuple** when… | Use a **list** when… |
|---|---|
| the number of items is fixed and each position has a meaning (a *record*) | you have a variable number of *similar* items |
| you need it as a dict key or set member | you need to append/remove/sort in place |
| you return several values from a function | you're building up a collection |

A useful rule of thumb: **lists are homogeneous collections, tuples are heterogeneous records.** Type hints reflect this: `list[int]` is "any number of ints", while `tuple[str, int]` is "exactly a str then an int". For a variable-length tuple of one type, write `tuple[int, ...]`.

In [ ]:
pair: tuple[str, int] = ("alice", 30)          # fixed shape
many: tuple[int, ...] = (1, 2, 3, 4)           # any length, all ints
print(pair, many)

## Summary

| Concept | C# | Python |
|---|---|---|
| Create | `(1, "a")` | `(1, "a")` or `1, "a"` |
| Single element | `ValueTuple.Create(1)` | `(1,)` — trailing comma! |
| Access | `t.Item1` | `t[0]` |
| Mutability | `ValueTuple` fields are mutable | always immutable (shallowly) |
| Return many values | `return (a, b);` | `return a, b` |
| Deconstruct | `var (a, b) = t;` | `a, b = t` |
| Composite dict key | `Dictionary<(int, int), T>` | `dict[tuple[int, int], T]` |
| Named elements | `(int X, int Y)` | `namedtuple("P", ["x", "y"])` |
| Typed record | `record struct P(int X, int Y)` | `class P(NamedTuple): x: int; y: int` |
| Copy with change | `p with { X = 1 }` | `p._replace(x=1)` |

## Exercises

1. Write a function `divmod_pair(a, b)` that returns the quotient and remainder as a tuple, then unpack the result into two variables.
2. Count how many times each `(x, y)` coordinate appears in `points = [(0, 0), (1, 2), (0, 0), (3, 4), (1, 2), (0, 0)]` using a dict with tuple keys.
3. Define a `typing.NamedTuple` called `Order` with fields `id: int`, `product: str`, `qty: int = 1`. Create an order, then create a copy with `qty=5`.
4. Predict the output: `t = ([1], [2]); t[0] += [3]`. What happens — and what does `t` contain afterwards? (Try it inside a `try`/`except`.)

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def divmod_pair(a, b):
    return a // b, a % b

q, r = divmod_pair(17, 5)    # 3, 2   (the built-in divmod() does the same)

# 2
points = [(0, 0), (1, 2), (0, 0), (3, 4), (1, 2), (0, 0)]
counts = {}
for p in points:
    counts[p] = counts.get(p, 0) + 1
# {(0, 0): 3, (1, 2): 2, (3, 4): 1}

# 3
from typing import NamedTuple

class Order(NamedTuple):
    id: int
    product: str
    qty: int = 1

o = Order(1, "widget")
bigger = o._replace(qty=5)

# 4 — a famous puzzle: it BOTH raises TypeError AND modifies the list.
#     `t[0] += [3]` first runs list.__iadd__ (mutating the list in place),
#     then tries to assign the result back into t[0], which fails.
t = ([1], [2])
try:
    t[0] += [3]
except TypeError as e:
    print(e)
print(t)    # ([1, 3], [2])
```

</details>